# Day 104 — Customer Churn Prediction using ANN (Keras and TensorFlow)

**Problem:** a bank wants to predict which customers will leave (churn). This is a **binary classification** problem on tabular data.

**Dataset:** the popular *Churn Modelling* bank dataset (`Churn_Modelling.csv`, 10,000 customers) from Kaggle.
Put the file next to this notebook. If it is not found, the notebook **generates a synthetic dataset with the same columns** so every cell still runs
(the numbers will then differ from the real data).

**Requirements:** `pip install tensorflow pandas scikit-learn matplotlib joblib`

**Workflow:**
1. Load and explore the data
2. Preprocess (drop IDs, encode categories, split, scale)
3. Build, compile and train an ANN with Keras
4. Plot training curves and evaluate (accuracy, confusion matrix, precision/recall, ROC AUC)
5. Compare with baselines (majority class, logistic regression, random forest)
6. Improve the model (dropout, early stopping, class weights, threshold tuning)
7. Predict for a new customer, then save and reload the model

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, precision_recall_curve, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)

## 1. Load the data

In [ ]:
def make_synthetic_churn(n=10000, seed=42):
    # Synthetic stand-in with the same columns as Churn_Modelling.csv
    rng = np.random.RandomState(seed)
    geography = rng.choice(["France", "Germany", "Spain"], size=n, p=[0.50, 0.25, 0.25])
    gender = rng.choice(["Female", "Male"], size=n, p=[0.45, 0.55])
    age = np.clip(rng.normal(39, 10, n), 18, 92).round().astype(int)
    products = rng.choice([1, 2, 3, 4], size=n, p=[0.51, 0.46, 0.025, 0.005])
    active = rng.binomial(1, 0.51, n)
    balance = np.where(rng.rand(n) < 0.36, 0.0, np.clip(rng.normal(120000, 30000, n), 0, None)).round(2)
    logit = (-1.35 + 0.1 * (age - 39) + 0.8 * (geography == "Germany") + 0.4 * (gender == "Female")
             - 1.0 * active - 1.0 * (products == 2) + 2.2 * (products >= 3) + 0.000004 * (balance - 60000))
    exited = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return pd.DataFrame({
        "RowNumber": np.arange(1, n + 1), "CustomerId": rng.randint(15_500_000, 15_900_000, n),
        "Surname": ["Customer"] * n,
        "CreditScore": np.clip(rng.normal(650, 97, n), 350, 850).round().astype(int),
        "Geography": geography, "Gender": gender, "Age": age, "Tenure": rng.randint(0, 11, n),
        "Balance": balance, "NumOfProducts": products, "HasCrCard": rng.binomial(1, 0.7, n),
        "IsActiveMember": active, "EstimatedSalary": rng.uniform(11, 200000, n).round(2), "Exited": exited})

CSV_PATH = "Churn_Modelling.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Loaded the real dataset:", CSV_PATH)
else:
    df = make_synthetic_churn()
    print("Churn_Modelling.csv not found -> using a SYNTHETIC dataset with the same columns")

df.head()

## 2. Explore the data

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:\n", df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else "none")
print("Duplicate rows:", df.duplicated().sum())
print("\nData types:\n", df.dtypes)
df.describe().T.round(2)

In [ ]:
churn_rate = df["Exited"].mean()
print(f"Churn rate: {churn_rate:.1%}  ->  a model that always predicts 'stays' gets {1 - churn_rate:.1%} accuracy")

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
df["Exited"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["tab:blue", "tab:red"])
axes[0].set_title("Class balance (0 = stayed, 1 = churned)"); axes[0].set_xticklabels(["stayed", "churned"], rotation=0)

df.groupby("Geography")["Exited"].mean().plot(kind="bar", ax=axes[1]); axes[1].set_title("Churn rate by Geography")
df.groupby("NumOfProducts")["Exited"].mean().plot(kind="bar", ax=axes[2]); axes[2].set_title("Churn rate by NumOfProducts")
for label, color in [(0, "tab:blue"), (1, "tab:red")]:
    axes[3].hist(df.loc[df["Exited"] == label, "Age"], bins=30, alpha=0.6, color=color, label=["stayed", "churned"][label])
axes[3].set_title("Age distribution"); axes[3].legend()
plt.tight_layout(); plt.show()

## 3. Preprocessing

- Drop identifier columns (`RowNumber`, `CustomerId`, `Surname`): they carry no predictive meaning
- One-hot encode `Geography` and `Gender` (drop the first level to avoid redundant columns)
- **Split first, then scale**: the scaler learns its mean and standard deviation from the training data only (prevents data leakage)
- Neural networks train much better on **scaled** inputs

In [ ]:
df_model = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
X = df_model.drop(columns="Exited")
y = df_model["Exited"].values.astype("float32")

categorical_cols = ["Geography", "Gender"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

# test set (kept untouched until the end), then a validation set carved out of the training data
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols),
], sparse_threshold=0)

X_train_s = preprocess.fit_transform(X_train).astype("float32")     # fit ONLY on the training data
X_val_s   = preprocess.transform(X_val).astype("float32")
X_test_s  = preprocess.transform(X_test).astype("float32")

print("Train / validation / test:", X_train_s.shape, X_val_s.shape, X_test_s.shape)
print("Feature names:", list(preprocess.get_feature_names_out()))
print("Churn rate in each split:", y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3))

## 4. Build the ANN with Keras

Architecture: `11 inputs → Dense(11, ReLU) → Dense(11, ReLU) → Dense(1, sigmoid)`

- Hidden layers: **ReLU**
- Output: **sigmoid** gives the probability of churn
- Loss: **binary cross entropy**, optimizer: **Adam**

In [ ]:
n_features = X_train_s.shape[1]

model = keras.Sequential([
    keras.Input(shape=(n_features,)),
    layers.Dense(11, activation="relu"),
    layers.Dense(11, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
], name="churn_ann")

model.summary()

In [ ]:
# Parameter count check: weights + biases per layer (Day 101 formula)
sizes = [n_features, 11, 11, 1]
print("Expected parameters:", sum(sizes[i] * sizes[i + 1] + sizes[i + 1] for i in range(len(sizes) - 1)))

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(X_train_s, y_train,
                    validation_data=(X_val_s, y_val),
                    epochs=100, batch_size=32, verbose=0)

print(f"Final epoch: train loss = {history.history['loss'][-1]:.4f}, val loss = {history.history['val_loss'][-1]:.4f}")
print(f"             train acc  = {history.history['accuracy'][-1]:.4f}, val acc  = {history.history['val_accuracy'][-1]:.4f}")

## 5. Training curves

- Loss falling on both curves means the model is learning.
- If **validation loss starts rising while training loss keeps falling**, the model is **overfitting**.

In [ ]:
def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train"); axes[0].plot(history.history["val_loss"], label="validation")
    axes[0].set_title(f"Loss {title}"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history.history["accuracy"], label="train"); axes[1].plot(history.history["val_accuracy"], label="validation")
    axes[1].set_title(f"Accuracy {title}"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_history(history, "(baseline ANN)")

## 6. Evaluate on the test set

Accuracy alone is misleading with imbalanced classes, so we also look at the **confusion matrix**, **precision/recall** and **ROC AUC**.

In [ ]:
def evaluate(name, proba, y_true, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    print(f"--- {name} (threshold = {threshold:.2f}) ---")
    print("Accuracy :", round(accuracy_score(y_true, pred), 4))
    print("ROC AUC  :", round(roc_auc_score(y_true, proba), 4))
    print("Confusion matrix [[TN FP] [FN TP]]:\n", cm)
    print(classification_report(y_true, pred, target_names=["stayed", "churned"], digits=3))
    return pred

proba_ann = model.predict(X_test_s, verbose=0).ravel()
pred_ann = evaluate("Baseline ANN", proba_ann, y_test)

In [ ]:
loss, acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"Keras evaluate -> test loss = {loss:.4f}, test accuracy = {acc:.4f}")
print(f"Majority-class baseline accuracy = {1 - y_test.mean():.4f}")

## 7. Compare with simple baselines

A neural network is not automatically the best choice for tabular data, so always compare with simpler models.

In [ ]:
log_reg = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_train_s, y_train)

results = {
    "Majority class": (np.zeros_like(y_test), np.zeros_like(y_test, dtype=float)),
    "Logistic regression": ((log_reg.predict_proba(X_test_s)[:, 1] >= 0.5).astype(int), log_reg.predict_proba(X_test_s)[:, 1]),
    "Random forest": ((rf.predict_proba(X_test_s)[:, 1] >= 0.5).astype(int), rf.predict_proba(X_test_s)[:, 1]),
    "ANN (baseline)": (pred_ann, proba_ann),
}

rows = []
for name, (pred, proba) in results.items():
    auc = roc_auc_score(y_test, proba) if name != "Majority class" else 0.5
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
                 "Recall (churn)": (pred[y_test == 1] == 1).mean(), "F1 (churn)": f1_score(y_test, pred), "ROC AUC": auc})
comparison = pd.DataFrame(rows).set_index("Model").round(3)
comparison

## 8. Improving the ANN

Ideas used below:
- **Bigger network + Dropout** (randomly switches off neurons while training, which reduces overfitting)
- **EarlyStopping** (stop when validation loss stops improving and restore the best weights)
- **Class weights** (give the rare "churned" class more importance)
- **Threshold tuning** (choose the probability cutoff using the validation set, not the test set)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def build_model(n_features, units=(64, 32), dropout=0.3):
    layer_list = [keras.Input(shape=(n_features,))]
    for u in units:
        layer_list += [layers.Dense(u, activation="relu"), layers.Dropout(dropout)]
    layer_list.append(layers.Dense(1, activation="sigmoid"))
    m = keras.Sequential(layer_list, name="churn_ann_improved")
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

weights = compute_class_weight("balanced", classes=np.array([0.0, 1.0]), y=y_train)
class_weight = {0: float(weights[0]), 1: float(weights[1])}
print("Class weights:", {k: round(v, 3) for k, v in class_weight.items()})

model2 = build_model(n_features)
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history2 = model2.fit(X_train_s, y_train, validation_data=(X_val_s, y_val),
                      epochs=200, batch_size=32, class_weight=class_weight,
                      callbacks=[early_stop], verbose=0)
print("Stopped after", len(history2.history["loss"]), "epochs")
plot_history(history2, "(improved ANN)")

In [ ]:
# Choose the decision threshold on the VALIDATION set (maximize F1 for the churn class)
proba_val = model2.predict(X_val_s, verbose=0).ravel()
precisions, recalls, thresholds = precision_recall_curve(y_val, proba_val)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-12)
best_threshold = float(thresholds[np.argmax(f1_scores)])
print(f"Best threshold on validation data (max F1): {best_threshold:.3f}")

proba_ann2 = model2.predict(X_test_s, verbose=0).ravel()
_ = evaluate("Improved ANN, default threshold", proba_ann2, y_test, 0.5)
pred_ann2 = evaluate("Improved ANN, tuned threshold", proba_ann2, y_test, best_threshold)

In [ ]:
results["ANN (improved, tuned threshold)"] = (pred_ann2, proba_ann2)
rows = []
for name, (pred, proba) in results.items():
    auc = roc_auc_score(y_test, proba) if name != "Majority class" else 0.5
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred),
                 "Recall (churn)": (pred[y_test == 1] == 1).mean(), "F1 (churn)": f1_score(y_test, pred), "ROC AUC": auc})
comparison = pd.DataFrame(rows).set_index("Model").round(3)
display(comparison) if "display" in globals() else print(comparison)

plt.figure(figsize=(6, 5))
for name in ["Logistic regression", "Random forest", "ANN (baseline)", "ANN (improved, tuned threshold)"]:
    fpr, tpr, _ = roc_curve(y_test, results[name][1])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, results[name][1]):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves on the test set"); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

## 9. Predict for a new customer

We pass a raw customer record through the **same fitted preprocessing** and then through the model.

In [ ]:
new_customer = pd.DataFrame([{
    "CreditScore": 600, "Geography": "Germany", "Gender": "Male", "Age": 40, "Tenure": 3,
    "Balance": 60000.0, "NumOfProducts": 2, "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 50000.0}])

x_new = preprocess.transform(new_customer).astype("float32")
p_new = float(model2.predict(x_new, verbose=0)[0, 0])
print(f"Probability of churn: {p_new:.3f} -> {'LIKELY TO CHURN' if p_new >= best_threshold else 'likely to stay'} (threshold {best_threshold:.2f})")

## 10. Save and reload

Save **both** the model and the preprocessing object, since new data must be transformed the same way.

In [ ]:
model2.save("churn_model.keras")
joblib.dump({"preprocess": preprocess, "threshold": best_threshold}, "churn_preprocess.joblib")

loaded_model = keras.models.load_model("churn_model.keras")
bundle = joblib.load("churn_preprocess.joblib")

p_loaded = float(loaded_model.predict(bundle["preprocess"].transform(new_customer).astype("float32"), verbose=0)[0, 0])
print("Prediction from the reloaded model:", round(p_loaded, 3), "| same as before:", np.isclose(p_loaded, p_new, atol=1e-5))

## Summary

| Step | Key point |
|---|---|
| Data | Tabular, imbalanced (about 20% churn in the real dataset) |
| Preprocessing | Drop IDs, one-hot encode, **split before scaling**, scale numeric features |
| Model | `Dense(ReLU) -> Dense(ReLU) -> Dense(sigmoid)`, Adam, binary cross entropy |
| Evaluation | Accuracy is not enough: use confusion matrix, recall, F1, ROC AUC |
| Baselines | Compare with majority class, logistic regression and random forest |
| Improvements | Dropout, early stopping, class weights, threshold tuning on validation data |
| Deployment | Save the model **and** the preprocessing |

## Practice exercises

1. Change the architecture (for example `(32, 16)` or `(128, 64, 32)`) and compare validation loss and ROC AUC.
2. Try `class_weight=None` with the improved model. How do recall and precision for churners change?
3. Replace Adam with SGD (`learning_rate=0.01`) and compare how fast the loss falls.
4. Add `keras.callbacks.ModelCheckpoint` to save the best model during training.
5. Use `sklearn.inspection.permutation_importance` on the random forest to see which features matter most.
6. Try a business-driven threshold: if contacting a likely churner costs 1 unit and losing a customer costs 10 units, which threshold minimizes the expected cost on the validation set?